# Hand tracking demo - MediaPipe (local runtime)

This demo reads a [LeRobot](https://docs.daft.ai/en/stable/datasets/lerobot/) dataset, runs hand tracking (MediaPipe) as a Daft UDF with `track_hands`, and shows the keypoints. Every method returns the same schema: a list of `{handedness, confidence, kp2d, kp3d?}` per frame (`kp3d` is null for MediaPipe).

## Setup

Install with `pip install daft-physical-ai[mediapipe]`, then import.

In [ ]:
from daft.datasets import lerobot

from daft_physical_ai import track_hands

## Configure

The dataset, the camera column to decode, and how many frames to run.

In [ ]:
DATASET = "pepijn223/egodex-test"
IMAGE_COLUMN = "observation.image"
LIMIT = 12

## Read the dataset

The LeRobot reader gives one row per frame, decoding the camera into an image column.

In [ ]:
df = lerobot.read(DATASET, load_video_frames=IMAGE_COLUMN).limit(LIMIT)

## Track hands

`track_hands` returns a hand-pose column. It's a lazy, batched Daft UDF, so nothing runs until we materialize below.

In [ ]:
df = df.with_column("hands", track_hands(df[IMAGE_COLUMN], method="mediapipe"))

## Inspect the results

`.show()` triggers execution and renders the keypoints per frame.

In [ ]:
df.select("episode_index", "frame_index", "hands").show()